# Napari Time-Projection Animation

Renders a single-cell time-projection animation by mapping each timepoint onto the Z-axis of a 3D napari viewer and screenshotting a reveal loop into an MP4.

**Requirements:** `napari`, `zarr`, `numpy`, `pandas`, `scipy`, `scikit-image`, `imageio`, `imageio-ffmpeg`, `tqdm`

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import zarr
import napari
import imageio.v2 as imageio
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter
from skimage.morphology import binary_dilation, disk

## 1. Configuration

In [ ]:
# --- Paths ---
sc_df_path   = '/path/to/sc_df.parquet'
image_path   = '/path/to/acquisition/{acq_id}.zarr'   # OME-NGFF image zarr
mask_path    = '/path/to/labels/{acq_id}.zarr'         # OME-NGFF labels zarr
output_path  = 'time_projection.mp4'
temp_dir     = '_tmp_frames'

# --- Target cell ---
target_id    = '330.3.5.PS0000'   # ID matching a row in sc_df
image_scale  = 5.04               # pixel size (µm) used to scale tracked coordinates
crop_padding = 250                # pixels of padding around the cell bounding box

# --- Rendering ---
z_scale               = 10        # Z spacing between timepoints
channel_colormaps     = ['green', 'magenta']
channel_contrast      = [(0, 2600), (0, 900)]   # (min, max) per channel
milestone_frames      = list(range(0, 75, 5))   # cyan flash every 5 frames
flash_decay_rate      = 0.15
baseline_opacity      = 0.40
camera_angles         = (1.31, -3.08, -51.21)   # set after manual inspection
camera_zoom           = 1.21
fps                   = 6
hold_final_seconds    = 3

## 2. Load data

In [ ]:
# Single-cell dataframe
df    = pd.read_parquet(sc_df_path)
sc_df = df[df['ID'] == target_id]

# Parse acquisition ID from target_id string  (format: cellID.row.col.exptID)
acq_id = f"({int(target_id.split('.')[1])}, {int(target_id.split('.')[2])})"

# OME-NGFF image — shape (T, C, Z, Y, X); max-project Z axis
zarr_group      = zarr.open(image_path.format(acq_id=acq_id), mode='r')
loaded_max_proj = zarr_group.images[...].max(axis=2)   # → (T, C, Y, X)

# OME-NGFF labels — shape (T, Y, X) or (T, Z, Y, X)
mask_store  = zarr.open(mask_path.format(acq_id=acq_id), mode='r')
label_name  = list(mask_store['labels'])[0]
segmentation = np.asarray(mask_store[f'labels/{label_name}/0'])
if segmentation.ndim == 4:
    segmentation = segmentation.max(axis=1)   # → (T, Y, X)

## 3. Crop to single cell and build focal mask

In [ ]:
frames   = sc_df['Frame'].values.astype(int)
x_coords = sc_df['y'].values * image_scale   # note: x/y swap is intentional (row/col vs y/x)
y_coords = sc_df['x'].values * image_scale

x_min, x_max = int(x_coords.min() - crop_padding), int(x_coords.max() + crop_padding)
y_min, y_max = int(y_coords.min() - crop_padding), int(y_coords.max() + crop_padding)

# Crop image and segmentation to the cell bounding box
sub_stack = loaded_max_proj[frames.min():frames.max()+1, :, x_min:x_max, y_min:y_max]
sub_seg   = segmentation[frames.min():frames.max()+1, x_min:x_max, y_min:y_max]

# Identify the target cell by its seg ID at the centre of the crop at each frame
mid_y, mid_x   = sub_seg.shape[1] // 2, sub_seg.shape[2] // 2
target_seg_ids = sub_seg[:, mid_y, mid_x]

# Build a binary focal mask: 1 where this cell is, 0 everywhere else
focal_mask = np.zeros_like(sub_seg, dtype=np.float32)
for t, sid in enumerate(target_seg_ids):
    if sid > 0:
        focal_mask[t] = (sub_seg[t] == sid).astype(np.float32)

# Mask the image stack so only the focal cell is visible
masked_stack = (sub_stack.astype(np.float32) * focal_mask[:, np.newaxis]).astype(sub_stack.dtype)

print(f'Cropped stack: {masked_stack.shape}  (T, C, Y, X)')

## 4. Pre-compute contour stack

A Gaussian-softened morphological gradient: dilate the blurred mask, subtract the original. The result is a float32 ring in `[0, 1]` that renders as a smooth glow in napari's additive mode.

In [ ]:
selem         = disk(2)
contour_stack = np.zeros_like(focal_mask, dtype=np.float32)

for t in tqdm(range(focal_mask.shape[0]), desc='Building contours'):
    if not np.any(focal_mask[t]):
        continue
    soft            = gaussian_filter((focal_mask[t] > 0).astype(np.float64), sigma=1.5)
    contour_stack[t] = binary_dilation(soft > 0.5, selem).astype(np.float32) - (soft > 0.5).astype(np.float32)

## 5. Set up napari viewer

Run this cell, adjust the camera manually in the viewer, then capture the state with `viewer.camera.dict()` and paste the values into the Configuration cell above before running the animation loop.

In [ ]:
T = masked_stack.shape[0]

viewer = napari.Viewer(title='Time Projection', ndisplay=3)

# Ghost: entire history at near-zero opacity — gives a faint silhouette of the full trajectory
viewer.add_image(masked_stack[:, 0], name='Ghost',
                 scale=(z_scale, 1, 1), colormap='green', opacity=0.02, blending='additive')

# History trail: grows one slice per frame at low opacity
mphi_hist = viewer.add_image(masked_stack[0:1, 0], name='Mphi_Hist',
                              scale=(z_scale, 1, 1), colormap='green', opacity=0.2, blending='additive')
mtb_hist  = viewer.add_image(masked_stack[0:1, 1], name='Mtb_Hist',
                              scale=(z_scale, 1, 1), colormap='magenta', opacity=0.2, blending='additive')

# Live cap: bright single-frame slice that rides the top of the tower
mphi_top  = viewer.add_image(masked_stack[0:1, 0], name='Mphi_Top',
                              colormap='green', blending='additive')
mtb_top   = viewer.add_image(masked_stack[0:1, 1], name='Mtb_Top',
                              colormap='magenta', blending='additive')

# Contour ring: follows the live cap
live_cont = viewer.add_image(contour_stack[0:1], name='Live_Cont',
                              colormap='cyan', blending='additive', opacity=0.99)
live_cont.contrast_limits = (0, 1)

# Initial camera
viewer.camera.center = (0, masked_stack.shape[2] / 2, masked_stack.shape[3] / 2)
viewer.camera.zoom   = camera_zoom
viewer.camera.angles = camera_angles

## 6. Render animation frames

In [ ]:
os.makedirs(temp_dir, exist_ok=True)
milestone_set = set(milestone_frames)

for i in tqdm(range(T), desc='Rendering'):

    # Grow history trail
    if i > 0:
        mphi_hist.data, mtb_hist.data = masked_stack[:i, 0], masked_stack[:i, 1]

    # Advance live cap and contour to frame i
    mphi_top.data = mtb_top.data = live_cont.data = None   # avoid stale reference
    mphi_top.data  = masked_stack[i:i+1, 0]
    mtb_top.data   = masked_stack[i:i+1, 1]
    live_cont.data = contour_stack[i:i+1]

    # Pin contrast limits to prevent per-frame auto-scaling flicker
    live_cont.contrast_limits         = (0, 1)
    mphi_top.contrast_limits          = channel_contrast[0]
    mtb_top.contrast_limits           = channel_contrast[1]
    mphi_hist.contrast_limits         = channel_contrast[0]
    mtb_hist.contrast_limits          = channel_contrast[1]

    # Translate live layers to the current Z position
    z = i * z_scale
    mphi_top.translate = mtb_top.translate = live_cont.translate = (z, 0, 0)

    # Deposit a glowing milestone marker
    if i in milestone_set:
        viewer.add_image(contour_stack[i:i+1], name=f'Mile_{i}',
                         colormap='cyan', opacity=1.0,
                         translate=(z, 0, 0), blending='additive')

    # Decay all active milestone glows toward baseline
    for lyr in viewer.layers:
        if lyr.name.startswith('Mile_') and lyr.opacity > baseline_opacity:
            lyr.opacity = max(baseline_opacity, lyr.opacity - flash_decay_rate)

    # Camera tracks the live cap
    viewer.camera.center = (z, viewer.camera.center[1], viewer.camera.center[2])

    for lyr in viewer.layers:
        lyr.refresh()
    viewer.screenshot(path=os.path.join(temp_dir, f'frame_{i:04d}.png'))

## 7. Compile to MP4

In [ ]:
pngs = sorted(glob.glob(os.path.join(temp_dir, 'frame_*.png')))

with imageio.get_writer(output_path, fps=fps, codec='libx264', quality=9) as w:
    for f in pngs:
        w.append_data(imageio.imread(f))
    final = imageio.imread(pngs[-1])
    for _ in range(fps * hold_final_seconds):
        w.append_data(final)

for f in pngs:
    os.remove(f)

print(f'Saved → {output_path}')